# CRMA Validation Against EM-DAT Flood Records

This notebook quantifies how well the CRMA risk model predicts historical
flood events recorded in the EM-DAT disaster database for East Africa (2010–2024).

**Metrics computed**:
- Hit Rate (HR) and False Alarm Ratio (FAR) at each risk threshold
- Brier Skill Score (BSS) vs. climatological baseline
- Peirce Skill Score (PSS = HR − FAR)
- ROC-AUC
- Confusion matrix at Orange/Red threshold

> **Note**: Populate the paths in section 6.0 before running.

In [ ]:
import numpy as np
import pandas as pd
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# --- Configure paths ---
RISK_GEOJSON_DIR = Path('data/risk_output')   # directory containing daily *.geojson files
EMDAT_CSV        = Path('data/emdat_east_africa.csv')

# Threshold for binary classification: >= this state is a 'hit'
HIT_THRESHOLD_STATE = 2  # 0=Green, 1=Yellow, 2=Orange, 3=Red

## 6.0  Load Risk GeoJSON Files

In [ ]:
def load_risk_geojsons(directory: Path) -> pd.DataFrame:
    records = []
    for fp in sorted(directory.glob('*.geojson')):
        try:
            fc = json.loads(fp.read_text())
            for feat in fc.get('features', []):
                p = feat['properties']
                records.append({
                    'date':           pd.Timestamp(fp.stem),
                    'admin1_pcode':   p.get('admin1_pcode'),
                    'risk_state':     p.get('risk_state', 0),
                    'risk_label':     p.get('risk_label', 'Green'),
                    'p_green':        p.get('p_green', 1.0),
                    'p_yellow':       p.get('p_yellow', 0.0),
                    'p_orange':       p.get('p_orange', 0.0),
                    'p_red':          p.get('p_red', 0.0),
                })
        except Exception:
            pass
    return pd.DataFrame(records)

risk_df = load_risk_geojsons(RISK_GEOJSON_DIR)
print(f'Loaded {len(risk_df)} admin-day risk records from {RISK_GEOJSON_DIR}')
if not risk_df.empty:
    print(risk_df.head())

## 6.1  Load EM-DAT Events

In [ ]:
def load_emdat(path: Path) -> pd.DataFrame:
    try:
        df = pd.read_csv(path)
        df['start_date'] = pd.to_datetime(df['start_date'])
        df['end_date']   = pd.to_datetime(df['end_date'])
        return df
    except FileNotFoundError:
        print(f'EM-DAT file not found: {path}')
        return pd.DataFrame()

emdat_df = load_emdat(EMDAT_CSV)
if not emdat_df.empty:
    print(f'Loaded {len(emdat_df)} EM-DAT flood events.')
    print(emdat_df[['country', 'admin1_pcode', 'start_date', 'deaths']].head())

## 6.2  Merge Risk Forecasts with EM-DAT Labels

For each admin-day in `risk_df`, label it as a positive event if an EM-DAT
flood was active in the same admin unit on that date.

In [ ]:
def label_events(risk_df: pd.DataFrame, emdat_df: pd.DataFrame) -> pd.DataFrame:
    if risk_df.empty or emdat_df.empty:
        return risk_df.assign(flood_observed=0)

    flood_days = set()
    for _, row in emdat_df.iterrows():
        dates = pd.date_range(row['start_date'], row['end_date'])
        for d in dates:
            flood_days.add((row['admin1_pcode'], d))

    risk_df = risk_df.copy()
    risk_df['flood_observed'] = [
        int((pcode, d) in flood_days)
        for pcode, d in zip(risk_df['admin1_pcode'], risk_df['date'])
    ]
    return risk_df

labelled = label_events(risk_df, emdat_df)
if not labelled.empty:
    pos = labelled['flood_observed'].sum()
    print(f'Positive admin-days: {pos} / {len(labelled)} ({100*pos/len(labelled):.2f}%)')

## 6.3  Skill Metrics

In [ ]:
def compute_metrics(df: pd.DataFrame, hit_state: int = 2) -> dict:
    if df.empty or 'flood_observed' not in df.columns:
        return {}

    observed  = df['flood_observed'].to_numpy(float)
    predicted = (df['risk_state'] >= hit_state).to_numpy(float)
    forecast_p = (df['p_orange'] + df['p_red']).to_numpy(float)  # P(high risk)

    tp = ((predicted == 1) & (observed == 1)).sum()
    fp = ((predicted == 1) & (observed == 0)).sum()
    fn = ((predicted == 0) & (observed == 1)).sum()
    tn = ((predicted == 0) & (observed == 0)).sum()

    hr  = tp / (tp + fn) if (tp + fn) > 0 else float('nan')
    far = fp / (tp + fp) if (tp + fp) > 0 else float('nan')
    pss = hr - (fp / (fp + tn)) if (fp + tn) > 0 else float('nan')

    bs       = float(np.mean((forecast_p - observed) ** 2))
    bs_clim  = float(np.mean((observed.mean() - observed) ** 2))
    bss      = 1 - bs / bs_clim if bs_clim > 0 else float('nan')

    return {
        'n_events': int(observed.sum()),
        'n_total':  len(df),
        'HR':  round(hr, 4), 'FAR': round(far, 4), 'PSS': round(pss, 4),
        'BS':  round(bs, 5), 'BSS': round(bss, 4),
        'TP': int(tp), 'FP': int(fp), 'FN': int(fn), 'TN': int(tn),
    }

metrics = compute_metrics(labelled, HIT_THRESHOLD_STATE)
if metrics:
    for k, v in metrics.items():
        print(f'  {k}: {v}')
else:
    print('No data — configure paths in section 6.0.')

## 6.4  ROC Curve (synthetic demo)

In [ ]:
import matplotlib.pyplot as plt

def plot_roc(forecast_p: np.ndarray, observed: np.ndarray, label: str = 'CRMA', ax=None):
    thresholds = np.linspace(0, 1, 101)
    hrs, fars = [], []
    for t in thresholds:
        pred = (forecast_p >= t).astype(float)
        tp = ((pred == 1) & (observed == 1)).sum()
        fp = ((pred == 1) & (observed == 0)).sum()
        fn = ((pred == 0) & (observed == 1)).sum()
        tn = ((pred == 0) & (observed == 0)).sum()
        hrs.append(tp / (tp + fn + 1e-9))
        fars.append(fp / (fp + tn + 1e-9))
    auc = float(-np.trapz(hrs, fars))
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 5))
    ax.plot(fars, hrs, label=f'{label} (AUC={auc:.3f})')
    ax.plot([0, 1], [0, 1], 'k--')
    ax.set_xlabel('False Alarm Rate')
    ax.set_ylabel('Hit Rate')
    ax.set_title('ROC Curve')
    ax.legend()
    return ax

# Synthetic demonstration (replace with real arrays from labelled dataframe)
rng = np.random.default_rng(1)
syn_obs  = rng.binomial(1, 0.08, 2000).astype(float)
syn_prob = np.clip(syn_obs * 0.6 + rng.uniform(0, 0.5, 2000) * 0.4, 0, 1)

fig, ax = plt.subplots(figsize=(5, 5))
plot_roc(syn_prob, syn_obs, 'CRMA (synthetic)', ax)
plt.tight_layout()
plt.show()